In [ ]:
import os
import yfinance as yf
import time
import math
import smtplib
from email.message import EmailMessage
import pandas as pd 

C = {'G': '\033[92m', 'R': '\033[91m', 'Y': '\033[93m', 'B': '\033[94m', 'C': '\033[96m', 'W': '\033[0m'}
ATR_MULTIPLIER = 2.0 
TRAILING_STOP_PCT = -0.015  
TP_MULTIPLIER = 3.0           
PARTIAL_TP_MULTIPLIER = 1.5   
COOLDOWN_MINUTES = 15         # Day 36: Hard lockout duration after any trade exit

PORTFOLIO_CAPITAL = 100000.0  
RISK_PER_TRADE_PCT = 0.01     

last_buy_price = 0
last_signal = 'None'
highest_price = 0
current_shares = 0  
take_profit_price = 0         
breakeven_active = False      
partial_profit_taken = False  
last_exit_time = 0            # Day 36: Stores the exact epoch time of our last exit

total_pnl = 0.0
wins, losses = 0, 0

def log_system_event(message):
    timestamp = time.ctime()
    with open('system_log.txt', 'a') as f:
        f.write(f'[{timestamp}], {message}\n')
    
def send_trade_notification(signal, price, pnl, shares):
    try:
        msg = EmailMessage()
        msg.set_content(f'Bot Alert: {signal}\nPrice: {price:.2f}\nShares: {shares}\nPnL: ₹{pnl:.2f}')
        msg['Subject'] = f'Trade Alert: {signal}'
        SENDER_EMAIL = 'smtp.quant@gmail.com'
        APP_PASSWORD = 'my passcode'
        msg['From'] = 'smtp.quant@gmail.com'
        msg['To'] = 'smtp.quant@gmail.com'
        
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login('smtp.quant@gmail.com','my passcode' )
            smtp.send_message(msg)
    except Exception as e:
        print(f"\n{C['R']} Email Notification Failed: {e}{C['W']}")

print(f"{C['B']} Initializing Quantitative Engine...{C['W']}")
master_df = yf.download('RELIANCE.NS', period='5d', interval='1m', progress=False)
print(f"{C['G']} System Online: Anti-Churn Cooldown Lockout Active ({COOLDOWN_MINUTES}m){C['W']}\n")

while True:
    try:
        # --- INCREMENTAL DATA FETCH ---
        new_data = yf.download('RELIANCE.NS', period='1d', interval='1m', progress=False)
        if not new_data.empty:
            master_df = pd.concat([master_df, new_data.tail(1)])
            master_df = master_df[~master_df.index.duplicated(keep='last')].tail(250)
        else:
            time.sleep(10); continue

        master_df['SMA_45'] = master_df['Close']['RELIANCE.NS'].rolling(window=45).mean()
        master_df['SMA_190'] = master_df['Close']['RELIANCE.NS'].rolling(window=190).mean()
        
        delta = master_df['Close']['RELIANCE.NS'].diff()
        master_df['RSI'] = 100 - (100 / (1 + (delta.clip(lower=0).rolling(14).mean() / delta.clip(upper=0).abs().rolling(14).mean())))
        
        master_df['EMA_12'] = master_df['Close']['RELIANCE.NS'].ewm(span=12, adjust=False).mean()
        master_df['EMA_26'] = master_df['Close']['RELIANCE.NS'].ewm(span=26, adjust=False).mean()
        master_df['MACD'] = master_df['EMA_12'] - master_df['EMA_26']
        master_df['MACD_Signal'] = master_df['MACD'].ewm(span=9, adjust=False).mean()

        high, low, pc = master_df['High']['RELIANCE.NS'], master_df['Low']['RELIANCE.NS'], master_df['Close']['RELIANCE.NS'].shift(1)
        tr = pd.concat([high-low, (high-pc).abs(), (low-pc).abs()], axis=1).max(axis=1)
        master_df['ATR'] = tr.rolling(window=14).mean()
        latest = master_df.iloc[-1]
        current_price = latest['Close']['RELIANCE.NS'].item()
        current_rsi = latest['RSI'].item()
        sma_45, sma_190 = latest['SMA_45'].item(), latest['SMA_190'].item()
        current_macd, current_macd_signal = latest['MACD'].item(), latest['MACD_Signal'].item()
        current_atr = latest['ATR'].item()

        if last_signal == 'BUY' and last_buy_price > 0 and current_shares > 0:
            
            stop_distance = current_atr * ATR_MULTIPLIER
            dynamic_stop_loss = last_buy_price - stop_distance
 
            if current_price >= (last_buy_price + current_atr) and not breakeven_active:
                breakeven_active = True
                print(f"\n{C['C']} BREAKEVEN SHIFT: Stop-Loss moved to entry (₹{last_buy_price:.2f}). Trade is RISK-FREE!{C['W']}")
            
            if breakeven_active:
                dynamic_stop_loss = last_buy_price

            partial_target = last_buy_price + (current_atr * PARTIAL_TP_MULTIPLIER)
            
            if current_price >= partial_target and not partial_profit_taken:
                partial_profit_taken = True 
                shares_to_sell = math.floor(current_shares / 2) 
                
                if shares_to_sell > 0:
                    pnl = (current_price - last_buy_price) * shares_to_sell
                    total_pnl += pnl
                    PORTFOLIO_CAPITAL += pnl
                    current_shares -= shares_to_sell 
                    
                    print(f"\n{C['G']} PARTIAL TP HIT! Sold {shares_to_sell} shares at {current_price:.2f}. Banked ₹{pnl:.2f}. Letting the rest run!{C['W']}")
                    
                    with open('trade_log.csv', 'a') as f:
                        f.write(f'{time.ctime()},{current_price},PARTIAL_TP,{pnl}\n')
                    send_trade_notification('PARTIAL_TP', current_price, pnl, shares_to_sell)
            
            if current_price > highest_price: highest_price = current_price
            
            trigger = None

            if current_price >= take_profit_price:
                trigger = 'TAKE_PROFIT'
            elif current_price <= dynamic_stop_loss: 
                trigger = 'BREAKEVEN_STOP' if breakeven_active else 'ATR_STOP_LOSS'
            elif (current_price - highest_price) / highest_price <= TRAILING_STOP_PCT: 
                trigger = 'TRAILING_STOP'
            
            if trigger:
                pnl = (current_price - last_buy_price) * current_shares
                total_pnl += pnl
                PORTFOLIO_CAPITAL += pnl 
                
                if pnl > 0: wins += 1 
                else: losses += 1
                
                color = C['G'] if (trigger == 'TAKE_PROFIT' or trigger == 'BREAKEVEN_STOP') else C['R']
                print(f"\n{color} {trigger} HIT! Sold final {current_shares} shares at {current_price:.2f} | PnL: ₹{pnl:.2f}{C['W']}")
                with open('trade_log.csv', 'a') as f:
                    f.write(f'{time.ctime()},{current_price},{trigger},{pnl}\n')
                send_trade_notification(trigger, current_price, pnl, current_shares)
                
                # Day 36: Stamp the exit time and reset variables
                last_exit_time = time.time()
                last_signal, last_buy_price, highest_price, current_shares, take_profit_price = 'SELL', 0, 0, 0, 0
                breakeven_active = False 
                partial_profit_taken = False
                time.sleep(60); continue

        if (sma_45 > sma_190) and (current_rsi < 60) and (current_macd > current_macd_signal):
            current_signal = 'BUY'
        else:
            current_signal = 'SELL'

        if current_signal != last_signal:
            if current_signal == 'SELL' and last_buy_price > 0 and current_shares > 0:
                pnl = (current_price - last_buy_price) * current_shares
                total_pnl += pnl
                PORTFOLIO_CAPITAL += pnl
                if pnl > 0: wins += 1 
                else: losses += 1
                print(f"\n{C['Y']} Trend Exit. Sold {current_shares} shares at {current_price:.2f} | PnL: ₹{pnl:.2f}{C['W']}")
                
                # Day 36: Stamp the exit time and reset variables
                last_exit_time = time.time()
                last_buy_price, highest_price, current_shares, take_profit_price = 0, 0, 0, 0
                breakeven_active = False
                partial_profit_taken = False
                
            elif current_signal == 'BUY':
                # Day 36: Calculate minutes passed since the last trade exit
                minutes_since_exit = (time.time() - last_exit_time) / 60
                
                if minutes_since_exit < COOLDOWN_MINUTES:
                    
                    pass
                else:
                    last_buy_price = highest_price = current_price
                    
                    risk_amount = PORTFOLIO_CAPITAL * RISK_PER_TRADE_PCT
                    stop_distance = current_atr * ATR_MULTIPLIER if current_atr > 0 else 0.5
                    current_shares = math.floor(risk_amount / stop_distance) 
                    
                    dynamic_stop_loss = last_buy_price - stop_distance
                    take_profit_price = last_buy_price + (current_atr * TP_MULTIPLIER)
                    
                    breakeven_active = False
                    partial_profit_taken = False
                    
                    print(f"\n{C['G']} ENTRY ALERT at {current_price:.2f}{C['W']}")
                    print(f"{C['C']}   ↳ Stop-Loss: ₹{dynamic_stop_loss:.2f} | Take-Profit Target: ₹{take_profit_price:.2f}{C['W']}")
                    
                    with open('trade_log.csv', 'a') as f:
                        f.write(f'{time.ctime()},{current_price},{current_signal},0\n')
                    send_trade_notification(current_signal, current_price, 0, current_shares)
            
            last_signal = current_signal

        pnl_val = (current_price - last_buy_price) * current_shares if last_buy_price > 0 else 0
        p_col = C['G'] if pnl_val >= 0 else C['R']
        
        mins_passed = (time.time() - last_exit_time) / 60
        if last_buy_price > 0:
            status_display = f"{C['C']}Target: ₹{take_profit_price:.2f}{C['W']}"
        elif mins_passed < COOLDOWN_MINUTES:
            status_display = f"{C['Y']} Lockout: {COOLDOWN_MINUTES - mins_passed:.1f}m left{C['W']}"
        else:
            status_display = f"{C['G']} Scanning Markets{C['W']}"
        
        print(f"\r{C['B']}[{time.strftime('%H:%M:%S')}] {C['W']}"
              f"Prc: {current_price:.2f} | "
              f"{status_display} | "
              f"Trade: {p_col}₹{pnl_val:+.2f}{C['W']} | "
              f"Acct: ₹{PORTFOLIO_CAPITAL:.2f} | "
              f"W/L: {wins}/{losses}    ", end="")

    except Exception as e:
        print(f"\n{C['R']} CRITICAL ERROR: {e}{C['W']}"); time.sleep(15)
    time.sleep(60)

 Initializing Quantitative Engine...
 System Online: Anti-Churn Cooldown Lockout Active (15m)

[15:06:28] Prc: 1302.80 |  Scanning Markets | Trade: ₹+0.00 | Acct: ₹100000.00 | W/L: 0/0    
 ENTRY ALERT at 1302.80
   ↳ Stop-Loss: ₹1301.63 | Take-Profit Target: ₹1304.56
[15:08:35] Prc: 1303.10 | Target: ₹1304.56 | Trade: ₹+255.84 | Acct: ₹100000.00 | W/L: 0/0    
 BREAKEVEN SHIFT: Stop-Loss moved to entry (₹1302.80). Trade is RISK-FREE!

 PARTIAL TP HIT! Sold 426 shares at 1305.10. Banked ₹979.77. Letting the rest run!

 TAKE_PROFIT HIT! Sold final 427 shares at 1305.10 | PnL: ₹982.07
[15:26:57] Prc: 1307.80 |  Scanning Markets | Trade: ₹+0.00 | Acct: ₹101961.84 | W/L: 1/0       
 ENTRY ALERT at 1307.00
   ↳ Stop-Loss: ₹1306.10 | Take-Profit Target: ₹1308.35
[15:29:04] Prc: 1306.40 | Target: ₹1308.35 | Trade: ₹-679.17 | Acct: ₹101961.84 | W/L: 1/0    
 BREAKEVEN SHIFT: Stop-Loss moved to entry (₹1307.00). Trade is RISK-FREE!

 PARTIAL TP HIT! Sold 566 shares at 1308.50. Banked ₹849.00. L

$RELIANCE.NS: possibly delisted; no price data found  (period=1d)

1 Failed download:
['RELIANCE.NS']: possibly delisted; no price data found  (period=1d)


[15:51:48] Prc: 1308.50 |  Scanning Markets | Trade: ₹+0.00 | Acct: ₹103659.84 | W/L: 2/0      

$RELIANCE.NS: possibly delisted; no price data found  (period=1d)

1 Failed download:
['RELIANCE.NS']: possibly delisted; no price data found  (period=1d)


[16:08:15] Prc: 1308.50 |  Scanning Markets | Trade: ₹+0.00 | Acct: ₹103659.84 | W/L: 2/0    